In [2]:
pip install psycopg2-binary

   ---------------------------------------- 0.0/2.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.7 MB ? eta -:--:--
   --- ------------------------------------ 0.3/2.7 MB ? eta -:--:--
   --- ------------------------------------ 0.3/2.7 MB ? eta -:--:--
   --- ------------------------------------ 0.3/2.7 MB ? eta -:--:--
   ------- -------------------------------- 0.5/2.7 MB 576.9 kB/s eta 0:00:04
   ------- -------------------------------- 0.5/2.7 MB 576.9 kB/s eta 0:00:04
   ----------- ---------------------------- 0.8/2.7 MB 551.3 kB/s eta 0:00:04
   ----------- ---------------------------- 0.8/2.7 MB 551.3 kB/s eta 0:00:04
   --------------- ------------------------ 1.0/2.7 MB 540.0 kB/s eta 0:00:04
   --------------- ------------------------ 1.0/2.7 MB 540.0 kB/s eta 0:00:04
   --------------- ------------------------ 1.0/2.7 MB 540.0 kB/s eta 0:00:04
   ------------------- -----------------

In [1]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta
import math

# Set random seed for reproducibility
np.random.seed(42)
random.seed(42)

# ------------------------------------------------------------
# 1. Define airports (IATA codes, coordinates, demand multipliers)
# ------------------------------------------------------------
airports = [
    {"code": "AUH", "city": "Abu Dhabi", "country": "UAE", "lat": 24.433, "lon": 54.651, "demand": 1.0},
    {"code": "LHR", "city": "London", "country": "UK", "lat": 51.470, "lon": -0.454, "demand": 1.5},
    {"code": "CDG", "city": "Paris", "country": "France", "lat": 49.009, "lon": 2.547, "demand": 1.3},
    {"code": "FRA", "city": "Frankfurt", "country": "Germany", "lat": 50.037, "lon": 8.562, "demand": 1.2},
    {"code": "SYD", "city": "Sydney", "country": "Australia", "lat": -33.946, "lon": 151.177, "demand": 1.4},
    {"code": "BOM", "city": "Mumbai", "country": "India", "lat": 19.089, "lon": 72.868, "demand": 1.3},
    {"code": "JFK", "city": "New York", "country": "USA", "lat": 40.641, "lon": -73.778, "demand": 1.6},
    {"code": "DXB", "city": "Dubai", "country": "UAE", "lat": 25.253, "lon": 55.364, "demand": 1.8},
    {"code": "IST", "city": "Istanbul", "country": "Turkey", "lat": 41.275, "lon": 28.752, "demand": 1.1},
    {"code": "SIN", "city": "Singapore", "country": "Singapore", "lat": 1.364, "lon": 103.991, "demand": 1.2},
    {"code": "HKG", "city": "Hong Kong", "country": "China", "lat": 22.308, "lon": 113.918, "demand": 1.3},
    {"code": "MEL", "city": "Melbourne", "country": "Australia", "lat": -37.673, "lon": 144.843, "demand": 1.2},
    {"code": "CAI", "city": "Cairo", "country": "Egypt", "lat": 30.112, "lon": 31.399, "demand": 0.9},
    {"code": "KHI", "city": "Karachi", "country": "Pakistan", "lat": 24.906, "lon": 67.082, "demand": 0.8},
    {"code": "CMB", "city": "Colombo", "country": "Sri Lanka", "lat": 7.181, "lon": 79.884, "demand": 0.7},
    {"code": "MLE", "city": "Male", "country": "Maldives", "lat": 4.192, "lon": 73.529, "demand": 0.8},
    {"code": "NRT", "city": "Tokyo", "country": "Japan", "lat": 35.765, "lon": 140.386, "demand": 1.1},
    {"code": "PEK", "city": "Beijing", "country": "China", "lat": 40.080, "lon": 116.584, "demand": 1.0},
    {"code": "MAD", "city": "Madrid", "country": "Spain", "lat": 40.498, "lon": -3.568, "demand": 1.0},
    {"code": "FCO", "city": "Rome", "country": "Italy", "lat": 41.800, "lon": 12.239, "demand": 1.0},
]

# Create a dictionary for easy access
airport_dict = {ap["code"]: ap for ap in airports}

# ------------------------------------------------------------
# 2. Define aircraft types with cabin capacities
# ------------------------------------------------------------
aircraft_types = [
    {"code": "A380", "name": "Airbus A380", "capacity": 517,
     "cabin_config": {"First": 14, "Business": 70, "Economy": 433}},
    {"code": "B787", "name": "Boeing 787 Dreamliner", "capacity": 290,
     "cabin_config": {"First": 0, "Business": 28, "Economy": 262}},
    {"code": "B777", "name": "Boeing 777-300ER", "capacity": 354,
     "cabin_config": {"First": 8, "Business": 42, "Economy": 304}},
    {"code": "A330", "name": "Airbus A330-300", "capacity": 300,
     "cabin_config": {"First": 0, "Business": 30, "Economy": 270}},
    {"code": "A320", "name": "Airbus A320", "capacity": 150,
     "cabin_config": {"First": 0, "Business": 16, "Economy": 134}},
    {"code": "A321", "name": "Airbus A321", "capacity": 185,
     "cabin_config": {"First": 0, "Business": 20, "Economy": 165}},
]

# ------------------------------------------------------------
# 3. Define fare classes with cabin and fare multipliers
# ------------------------------------------------------------
# Typical fare classes in airline revenue management
# Cabin: First (F, A), Business (J, C, D, I), Economy (Y, B, M, H, Q, V, W, S, T, L, K)
# For simplicity, we define a representative set
fare_classes = [
    {"code": "F", "cabin": "First", "multiplier": 1.0},   # First full fare
    {"code": "A", "cabin": "First", "multiplier": 0.8},   # First discounted
    {"code": "J", "cabin": "Business", "multiplier": 1.0},# Business full fare
    {"code": "C", "cabin": "Business", "multiplier": 0.8},
    {"code": "D", "cabin": "Business", "multiplier": 0.7},
    {"code": "I", "cabin": "Business", "multiplier": 0.6},
    {"code": "Y", "cabin": "Economy", "multiplier": 1.0}, # Economy full fare
    {"code": "B", "cabin": "Economy", "multiplier": 0.9},
    {"code": "M", "cabin": "Economy", "multiplier": 0.8},
    {"code": "H", "cabin": "Economy", "multiplier": 0.7},
    {"code": "Q", "cabin": "Economy", "multiplier": 0.6},
    {"code": "V", "cabin": "Economy", "multiplier": 0.5},
    {"code": "W", "cabin": "Economy", "multiplier": 0.4},
    {"code": "S", "cabin": "Economy", "multiplier": 0.3},
]

# ------------------------------------------------------------
# 4. Helper functions
# ------------------------------------------------------------
def haversine_distance(lat1, lon1, lat2, lon2):
    """Compute great-circle distance in km between two points."""
    R = 6371  # Earth radius in km
    phi1 = math.radians(lat1)
    phi2 = math.radians(lat2)
    delta_phi = math.radians(lat2 - lat1)
    delta_lambda = math.radians(lon2 - lon1)
    a = math.sin(delta_phi/2)**2 + math.cos(phi1)*math.cos(phi2)*math.sin(delta_lambda/2)**2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1-a))
    return R * c

def flight_duration_hours(distance_km):
    """Estimate flight duration based on distance (cruise speed ~900 km/h + 1h buffer)."""
    return distance_km / 850.0 + 0.75  # hours

def generate_flight_number():
    """Random Etihad flight number (EY 100-999)."""
    return f"EY{random.randint(100, 999)}"

def assign_aircraft(distance_km, demand_factor):
    """Choose aircraft type based on distance and demand."""
    # Long haul (>4000km) -> wide-body; short haul -> narrow-body
    if distance_km > 4000:
        choices = ["A380", "B777", "B787"]
        # A380 for very high demand, B777/B787 for others
        if demand_factor > 1.3:
            return "A380"
        else:
            return random.choice(["B777", "B787"])
    elif distance_km > 2000:
        # Medium haul
        return random.choice(["B787", "A330"])
    else:
        # Short haul
        return random.choice(["A320", "A321"])

# ------------------------------------------------------------
# 5. Generate flights
# ------------------------------------------------------------
start_date = datetime(2024, 1, 1)
end_date = datetime(2024, 1, 31)  # 31 days
date_range = [start_date + timedelta(days=i) for i in range((end_date - start_date).days + 1)]

flights = []
flight_id = 1

# We'll generate flights from AUH to all other airports and also return flights.
# For each destination, we decide frequency per day based on demand.
# For simplicity, we'll generate for each date and each destination (excluding AUH) with probability proportional to demand.

for dep_date in date_range:
    for dest_ap in airports:
        if dest_ap["code"] == "AUH":
            continue
        # Probability of having a flight on this day (e.g., demand factor * 0.3, at most 1)
        prob = min(dest_ap["demand"] * 0.3, 0.9)
        if random.random() < prob:
            # Determine direction: from AUH to dest or dest to AUH? We'll generate both with equal chance.
            # But to keep schedule, we'll generate both directions on the same day? That might be unrealistic.
            # Better: generate flight from AUH to dest, and also from dest to AUH on the same day (if demand high)
            # We'll use a separate probability for return flight.
            # For simplicity, we'll generate one flight per day per direction with independent probability.
            # Direction AUH -> dest
            if random.random() < 0.6:  # 60% chance for outbound
                origin = "AUH"
                dest = dest_ap["code"]
                dep_time = f"{random.randint(0, 23):02d}:{random.randint(0, 59):02d}"
                # Calculate distance and flight time
                dist = haversine_distance(
                    airport_dict[origin]["lat"], airport_dict[origin]["lon"],
                    airport_dict[dest]["lat"], airport_dict[dest]["lon"]
                )
                duration = flight_duration_hours(dist)
                arr_time_dt = datetime.strptime(dep_time, "%H:%M") + timedelta(hours=duration)
                arr_time = arr_time_dt.strftime("%H:%M")
                # Assign aircraft based on distance and demand
                ac_type = assign_aircraft(dist, dest_ap["demand"])
                # Get capacity from aircraft_types
                capacity = next(ac["capacity"] for ac in aircraft_types if ac["code"] == ac_type)
                flights.append({
                    "flight_id": flight_id,
                    "flight_number": generate_flight_number(),
                    "origin": origin,
                    "dest": dest,
                    "dep_date": dep_date.strftime("%Y-%m-%d"),
                    "dep_time": dep_time,
                    "arr_time": arr_time,
                    "aircraft": ac_type,
                    "capacity": capacity
                })
                flight_id += 1

            # Direction dest -> AUH
            if random.random() < 0.5:  # 50% chance for inbound (some routes have fewer returns)
                origin = dest_ap["code"]
                dest = "AUH"
                dep_time = f"{random.randint(0, 23):02d}:{random.randint(0, 59):02d}"
                dist = haversine_distance(
                    airport_dict[origin]["lat"], airport_dict[origin]["lon"],
                    airport_dict[dest]["lat"], airport_dict[dest]["lon"]
                )
                duration = flight_duration_hours(dist)
                arr_time_dt = datetime.strptime(dep_time, "%H:%M") + timedelta(hours=duration)
                arr_time = arr_time_dt.strftime("%H:%M")
                ac_type = assign_aircraft(dist, airport_dict[origin]["demand"])  # use origin demand
                capacity = next(ac["capacity"] for ac in aircraft_types if ac["code"] == ac_type)
                flights.append({
                    "flight_id": flight_id,
                    "flight_number": generate_flight_number(),
                    "origin": origin,
                    "dest": dest,
                    "dep_date": dep_date.strftime("%Y-%m-%d"),
                    "dep_time": dep_time,
                    "arr_time": arr_time,
                    "aircraft": ac_type,
                    "capacity": capacity
                })
                flight_id += 1

print(f"Generated {len(flights)} flights.")

# Convert to DataFrame
flights_df = pd.DataFrame(flights)

# ------------------------------------------------------------
# 6. Generate fares for each flight and class
# ------------------------------------------------------------
# We need base fare per cabin per flight. Base fare depends on distance and demand.
# For economy Y: base = 0.1 * distance + 50 (in USD) plus random variation.
# Business base = 3 * economy Y, First base = 2 * business base (or 6*Y)
# Then for each class in cabin, fare = base * multiplier.

fares_list = []

# Pre-compute distance for each flight (since we need it multiple times)
flight_distances = {}
for idx, row in flights_df.iterrows():
    origin = row["origin"]
    dest = row["dest"]
    dist = haversine_distance(
        airport_dict[origin]["lat"], airport_dict[origin]["lon"],
        airport_dict[dest]["lat"], airport_dict[dest]["lon"]
    )
    flight_distances[row["flight_id"]] = dist

for flight_id in flights_df["flight_id"]:
    dist = flight_distances[flight_id]
    # Get demand factor for destination (or origin depending on direction)
    # Use average demand of origin and dest? We'll use dest demand for simplicity.
    dest_code = flights_df[flights_df["flight_id"]==flight_id]["dest"].values[0]
    demand_factor = airport_dict[dest_code]["demand"]
    
    # Base fares with some randomness (±10%)
    base_y = (0.1 * dist + 50) * (1 + 0.1 * random.uniform(-1, 1))
    base_j = base_y * 3.0 * (1 + 0.1 * random.uniform(-1, 1))
    base_f = base_j * 2.0 * (1 + 0.1 * random.uniform(-1, 1))  # First ~6x Y
    
    for fc in fare_classes:
        cabin = fc["cabin"]
        if cabin == "First":
            base = base_f
        elif cabin == "Business":
            base = base_j
        else:
            base = base_y
        fare = base * fc["multiplier"]
        # Add some flight-specific adjustment based on demand
        fare *= (0.9 + 0.2 * demand_factor)  # higher demand -> higher fare
        fares_list.append({
            "flight_id": flight_id,
            "class_code": fc["code"],
            "fare_amount": round(fare, 2)
        })

fares_df = pd.DataFrame(fares_list)

# ------------------------------------------------------------
# 7. Generate booking curves
# ------------------------------------------------------------
# We'll generate cumulative bookings at specific days before departure (DCPs)
dcp_days = [330, 300, 270, 240, 210, 180, 150, 120, 90, 60, 30, 14, 7, 3, 1, 0]  # days before departure

# For each flight, we need to allocate target demand per class based on cabin capacities.
# First, get cabin capacities for each aircraft type
aircraft_capacity = {ac["code"]: ac["cabin_config"] for ac in aircraft_types}

# For each flight and class, generate total target bookings (final cumulative bookings at dcp=0)
# Then generate booking curve using logistic function.

bookings_list = []

# We'll also need a mapping from class code to cabin
class_to_cabin = {fc["code"]: fc["cabin"] for fc in fare_classes}

for flight_id in flights_df["flight_id"]:
    flight_row = flights_df[flights_df["flight_id"]==flight_id].iloc[0]
    ac_type = flight_row["aircraft"]
    cabin_caps = aircraft_capacity[ac_type]  # dict: cabin -> seats
    # For each cabin, we have multiple fare classes. We'll distribute cabin capacity among classes
    # using random proportions that sum to 1 for that cabin.
    # Then target bookings for each class = proportion * cabin_capacity * (some random factor between 0.7 and 1.2)
    # This allows oversell up to 20% above capacity.
    
    # Group fare classes by cabin
    classes_by_cabin = {}
    for fc in fare_classes:
        cabin = fc["cabin"]
        if cabin not in classes_by_cabin:
            classes_by_cabin[cabin] = []
        classes_by_cabin[cabin].append(fc["code"])
    
    # For each cabin that exists in this aircraft, generate target bookings for its classes
    cabin_targets = {}
    for cabin, seat_count in cabin_caps.items():
        if seat_count == 0:
            continue
        class_list = classes_by_cabin.get(cabin, [])
        if not class_list:
            continue
        # Random proportions (Dirichlet-like)
        props = np.random.dirichlet(np.ones(len(class_list)))  # sum to 1
        # Multiply by seat_count and a random load factor (0.7 to 1.2)
        load_factor = random.uniform(0.7, 1.2)
        for i, class_code in enumerate(class_list):
            target = props[i] * seat_count * load_factor
            cabin_targets[class_code] = target
    
    # Now for each class, generate booking curve
    for class_code, target in cabin_targets.items():
        # Logistic curve parameters: target is asymptote.
        # We want bookings to start near 0 at 330 days out and reach target at departure.
        # Use logistic function: bookings(t) = target / (1 + exp(-k*(t - t0)))
        # Where t is days before departure (t=330 early, t=0 departure). We need decreasing function: as t decreases, bookings increase.
        # So let x = days_before (from 330 down to 0). We want bookings to increase from near 0 to target.
        # A simple approach: bookings(t) = target * (1 - exp(-lambda * (330 - t))) but that's exponential.
        # We'll use logistic: bookings(t) = target / (1 + exp(growth_rate * (t - midpoint)))
        # Choose midpoint such that at t=330, bookings are small; at t=0, bookings near target.
        # Let growth_rate = 0.03, midpoint = 100. Then at t=330, exp(0.03*(330-100))=exp(6.9)=~1000, so bookings ~ target/1001 ~ 0.
        # At t=0, exp(0.03*(0-100))=exp(-3)=0.05, so bookings ~ target/1.05 ~ 0.95 target. Slight under.
        # We can adjust.
        growth_rate = 0.03
        midpoint = 100
        # For each dcp, compute cumulative bookings
        for dcp in dcp_days:
            # t is days before departure (dcp)
            log_arg = math.exp(growth_rate * (dcp - midpoint))
            cum_book = target / (1 + log_arg)
            # Add some random noise (but ensure monotonicity across dcp)
            # We'll add noise after generating all points, but for simplicity we'll add small noise now.
            # However, noise could break monotonicity. We'll add noise that is proportional to cum_book and ensure it doesn't decrease.
            # Better: generate cumulative bookings for all dcp in decreasing order (from high dcp to low) and then add noise that increases.
            # We'll handle by sorting later.
            # For now, we'll store raw logistic value and then apply monotonic noise later.
            bookings_list.append({
                "flight_id": flight_id,
                "class_code": class_code,
                "days_before": dcp,
                "cumulative_bookings": round(cum_book, 2)
            })

# Convert to DataFrame
bookings_df = pd.DataFrame(bookings_list)

# Now we need to ensure monotonicity: as days_before decreases (closer to departure), cumulative_bookings should not decrease.
# Since we used logistic with positive growth rate, it is monotonic increasing as dcp decreases (because exp term decreases).
# However, with random variation we might have introduced non-monotonicity? We didn't add noise yet.
# Let's add small random noise that preserves monotonicity.
# For each flight/class group, sort by days_before descending (from 330 to 0). Then add noise that is cumulative sum of positive increments.
# Approach: For each group, get sorted data. Then generate random increments for each step (positive), and cumulative sum from the logistic baseline.
# Or we can multiply the logistic value by random factor between 0.95 and 1.05, but ensure the product does not decrease.
# We'll do a simple method: for each group, after sorting, we compute the logistic baseline, then add a random percentage change, but then enforce that each subsequent value is >= previous.
# This might clip some values.

# Group by flight and class
grouped = bookings_df.groupby(['flight_id', 'class_code'])
new_rows = []
for (fid, cc), group in grouped:
    group_sorted = group.sort_values('days_before', ascending=False).copy()  # from 330 down to 0
    base_values = group_sorted['cumulative_bookings'].values
    # Add random noise: multiply by factor between 0.97 and 1.03
    noisy = base_values * np.random.uniform(0.97, 1.03, size=len(base_values))
    # Enforce monotonic non-decreasing as days_before decreases
    for i in range(1, len(noisy)):
        if noisy[i] < noisy[i-1]:
            noisy[i] = noisy[i-1]  # set to previous (can also add small increment)
    group_sorted['cumulative_bookings'] = np.round(noisy, 2)
    new_rows.append(group_sorted)

bookings_df = pd.concat(new_rows, ignore_index=True)

# ------------------------------------------------------------
# 8. Output to CSV files
# ------------------------------------------------------------
flights_df.to_csv('flights.csv', index=False)
fares_df.to_csv('fares.csv', index=False)
# Also output fare_classes reference
fare_classes_df = pd.DataFrame(fare_classes)
fare_classes_df.to_csv('fare_classes.csv', index=False)
bookings_df.to_csv('bookings_cumulative.csv', index=False)

print("Dataset generation complete. Files saved:")
print("- flights.csv")
print("- fares.csv")
print("- fare_classes.csv")
print("- bookings_cumulative.csv")

Generated 223 flights.
Dataset generation complete. Files saved:
- flights.csv
- fares.csv
- fare_classes.csv
- bookings_cumulative.csv
